# Full CASIA Training and Evaluation (Google Colab)

Run top to bottom with **Runtime → Change runtime type → T4 GPU**. This notebook trains the project Custom CNN and MobileNetV2 on CASIA v1.0 and v2.0, reports validation metrics during training, and evaluates both models on validation and held-out test sets.

## Dataset setup

1. For this multi-GB file, upload the ZIP with Colab’s **Files** sidebar (folder icon → Upload) into `/content`; then run the upload cell. This avoids keeping the archive bytes in a notebook variable. If it is not already in `/content`, the cell offers the standard file picker.
2. The ZIP should contain `casia_v1/` and `casia_v2/` at its root. Upload time still depends on the actual connection to Colab.
3. The notebook creates a reproducible group-aware 70/15/15 split with the repository's filename labels (`Au_` authentic, `Tp_` manipulated). The test split is reserved until training and checkpoint selection are complete.

To reduce runtime, the notebook caches resized images on the Colab VM after the first read, skips the redundant full image verification pass, and uses shorter default epoch limits with early stopping. Models, plots, metrics, and a CSV summary are saved to `/content/training_outputs` and downloaded as a ZIP at the end.


## 1. Connect Drive and set the dataset ZIP path

In [ ]:
from google.colab import files
from pathlib import Path

# Prefer a ZIP uploaded with the Colab Files sidebar. It should appear in /content.
zip_candidates = sorted(Path("/content").glob("*.zip"))
if len(zip_candidates) == 1:
    DATASET_ZIP = zip_candidates[0]
elif len(zip_candidates) > 1:
    raise ValueError(f"More than one ZIP is in /content; keep only the dataset ZIP or set DATASET_ZIP manually: {zip_candidates}")
else:
    print("No ZIP found in /content. Choose the dataset ZIP in the file picker.")
    uploaded_files = files.upload()
    zip_names = [name for name in uploaded_files if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(f"Upload exactly one dataset ZIP file; received: {list(uploaded_files)}")
    DATASET_ZIP = Path("/content") / zip_names[0]
    del uploaded_files
if not DATASET_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATASET_ZIP}")
print(f"Dataset archive: {DATASET_ZIP} ({DATASET_ZIP.stat().st_size / (1024**3):.2f} GiB)")


## 2. Get project code, install dependencies, and verify GPU

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/Salman-ahmed-2/image-manipulation-detection.git"
REPO_DIR = Path("/content/image-manipulation-detection")
if not (REPO_DIR / "config.py").is_file():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
import importlib.util
required_packages = {
    "pandas": "pandas", "sklearn": "scikit-learn", "PIL": "Pillow",
    "matplotlib": "matplotlib", "dotenv": "python-dotenv",
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if not gpus: raise RuntimeError("No GPU found. In Colab choose Runtime → Change runtime type → GPU, then rerun.")
for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
print("GPU:", gpus)


## 3. Extract dataset and create the project split

In [ ]:
import zipfile
import pandas as pd
from config import Config
from src.data.dataset_loader import extract_label_from_filename
from src.data.dataset_splitter import DatasetSplitter

RAW_DIR = REPO_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as archive:
    names = archive.namelist()
    roots = {name.split("/", 1)[0] for name in names if "/" in name}
    if not {"casia_v1", "casia_v2"}.issubset(roots):
        raise ValueError("Dataset ZIP must contain casia_v1/ and casia_v2/ at its root.")
    for name in names:
        target = (RAW_DIR / name).resolve()
        if target != RAW_DIR.resolve() and RAW_DIR.resolve() not in target.parents:
            raise ValueError(f"Unsafe path in dataset ZIP: {name}")
    print(f"Extracting {len(names):,} archive entries to the Colab local disk...", flush=True)
    for index, name in enumerate(names, start=1):
        archive.extract(name, RAW_DIR)
        if index % 500 == 0 or index == len(names):
            print(f"Extracted {index:,}/{len(names):,} entries", flush=True)

# Build labels and a group-aware split from filenames without opening every image
# once for verification and then a second time for training.
records = []
unrecognized_prefixes = 0
for dataset_dir, dataset_version in ((Config.CASIA_V1_DIR, Config.CASIA_V1_VERSION),
                                     (Config.CASIA_V2_DIR, Config.CASIA_V2_VERSION)):
    paths = sorted(path for path in dataset_dir.rglob("*")
                   if path.is_file() and path.suffix.lower() in Config.DATASET_IMAGE_EXTENSIONS)
    print(f"Found {len(paths):,} supported image files in {dataset_dir.name}")
    for path in paths:
        # Use only the instructed filename prefixes: Au_ = authentic, Tp_ = tampered.
        if not path.name.startswith((Config.AUTHENTIC_PREFIX, Config.TAMPERED_PREFIX)):
            unrecognized_prefixes += 1
            continue
        try:
            label_info = extract_label_from_filename(path.name)
        except ValueError:
            continue
        records.append({
            "image_id": f"{dataset_version}:{path.relative_to(dataset_dir).as_posix()}",
            "filepath": str(path.resolve()),
            "filename": path.name,
            "label": label_info["label"],
            "label_name": label_info["label_name"],
            "group_id": label_info["group_id"],
            "dataset_version": dataset_version,
        })
if unrecognized_prefixes:
    print(f"Warning: skipped {unrecognized_prefixes:,} images without Au_/Tp_ filename prefixes.")
if not records:
    raise ValueError("No supported, labeled CASIA images were found in the uploaded archive.")
split_df = DatasetSplitter(random_state=Config.RANDOM_SEED).split_dataframe(pd.DataFrame(records))
split_df.to_csv(Config.DATASET_SPLIT_PATH, index=False)
expected = {"train", "validation", "test"}
if set(split_df["split"].unique()) != expected:
    raise ValueError(f"Expected all splits; found {sorted(split_df['split'].unique())}")
display(pd.crosstab(split_df["split"], split_df["label"], margins=True))


## 4. Build streaming TensorFlow input pipelines

In [ ]:
import numpy as np
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
IMAGE_SIZE = tuple(Config.TARGET_SIZE)
BATCH_SIZE, SEED = Config.BATCH_SIZE, Config.RANDOM_SEED
# Lower limits shorten the first full run; increase them for longer training.
CNN_EPOCHS = 20
MOBILENET_WARMUP_EPOCHS = 6
MOBILENET_FINE_TUNE_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 4
CACHE_DIR = Path("/content/casia_resized_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

def decode_resize(path_tensor):
    def load_image(path_value):
        with Image.open(path_value.decode("utf-8")) as image:
            image = image.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            return np.asarray(image, dtype=np.uint8)
    image = tf.numpy_function(load_image, [path_tensor], tf.uint8)
    image.set_shape((IMAGE_SIZE[1], IMAGE_SIZE[0], 3))
    return image

def make_dataset(frame, normalization, training=False):
    split_name = str(frame["split"].iloc[0])
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["filepath"].astype(str).to_numpy(),
        frame["label"].to_numpy(dtype=np.int32),
    ))
    dataset = dataset.map(lambda path, label: (decode_resize(path), label),
                          num_parallel_calls=tf.data.AUTOTUNE)
    cache_file = CACHE_DIR / f"{split_name}_224_uint8.cache"
    dataset = dataset.cache(str(cache_file))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def normalize(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        if normalization == "mobilenet":
            image = image * 2.0 - 1.0
        return image, tf.cast(label, tf.float32)
    return dataset.map(normalize, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df = split_df[split_df.split == "train"].reset_index(drop=True)
val_df = split_df[split_df.split == "validation"].reset_index(drop=True)
test_df = split_df[split_df.split == "test"].reset_index(drop=True)
for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    if frame.empty or set(frame.label.unique()) != {0, 1}:
        raise ValueError(f"{name} split must contain both classes; got {frame.label.value_counts().to_dict()}")
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df.label.to_numpy())
CLASS_WEIGHTS = {int(label): float(weight) for label, weight in zip(classes, weights)}
cache_gib = len(split_df) * IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3 / (1024**3)
print("Class weights:", CLASS_WEIGHTS)
print(f"Split sizes: train={len(train_df):,}, validation={len(val_df):,}, test={len(test_df):,}")
print(f"First pass creates about {cache_gib:.1f} GiB of uint8 image cache; later epochs reuse it.")


## 5. Shared training and evaluation helpers

In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
OUTPUT_DIR = Path("/content/training_outputs")
for subdir in ("models", "plots", "metrics"): (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)
def get_callbacks(model_key, patience=EARLY_STOPPING_PATIENCE):
    return [tf.keras.callbacks.ModelCheckpoint(str(OUTPUT_DIR / "models" / f"{model_key}_best.keras"), monitor="val_loss", save_best_only=True, verbose=1), tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1), tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)]
def evaluate_split(model, frame, normalization, split_name):
    probabilities = model.predict(make_dataset(frame, normalization), verbose=1).reshape(-1)
    labels = frame.label.to_numpy(dtype=np.int32)
    predictions = (probabilities >= Config.CLASSIFICATION_THRESHOLD).astype(np.int32)
    metrics = {"n": int(len(labels)), "accuracy": float(accuracy_score(labels, predictions)), "precision": float(precision_score(labels, predictions, zero_division=0)), "recall": float(recall_score(labels, predictions, zero_division=0)), "f1_score": float(f1_score(labels, predictions, zero_division=0)), "roc_auc": float(roc_auc_score(labels, probabilities)), "threshold": float(Config.CLASSIFICATION_THRESHOLD), "confusion_matrix_labels_0_1": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(), "classification_report": classification_report(labels, predictions, target_names=["Authentic", "Manipulated"], output_dict=True, zero_division=0)}
    print(f"\n{split_name.upper()} RESULTS — {len(labels):,} images")
    print(classification_report(labels, predictions, target_names=["Authentic", "Manipulated"], zero_division=0))
    print(f"Accuracy {metrics['accuracy']:.4f} | Precision {metrics['precision']:.4f} | Recall {metrics['recall']:.4f} | F1 {metrics['f1_score']:.4f} | ROC-AUC {metrics['roc_auc']:.4f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:", metrics["confusion_matrix_labels_0_1"])
    return metrics
def save_history(histories, model_key):
    merged = {}
    for history in histories:
        for key, values in history.history.items(): merged.setdefault(key, []).extend([float(v) for v in values])
    pd.DataFrame(merged).to_csv(OUTPUT_DIR / "metrics" / f"{model_key}_history.csv", index_label="epoch")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key in ("accuracy", "val_accuracy"):
        if key in merged: axes[0].plot(merged[key], label=key)
    for key in ("loss", "val_loss"):
        if key in merged: axes[1].plot(merged[key], label=key)
    axes[0].set_title("Accuracy"); axes[1].set_title("Loss")
    for axis in axes: axis.set_xlabel("Epoch"); axis.legend(); axis.grid(True, alpha=.3)
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / "plots" / f"{model_key}_training_curves.png", dpi=150); plt.show()


## 6. Train the Custom CNN

In [ ]:
from src.models.cnn_model import build_custom_cnn
cnn = build_custom_cnn(input_shape=Config.INPUT_SHAPE)
cnn_history = cnn.fit(make_dataset(train_df, "standard", training=True), validation_data=make_dataset(val_df, "standard"), epochs=CNN_EPOCHS, callbacks=get_callbacks("custom_cnn"), class_weight=CLASS_WEIGHTS, verbose=1)
cnn = tf.keras.models.load_model(OUTPUT_DIR / "models" / "custom_cnn_best.keras")
cnn.save(OUTPUT_DIR / "models" / "custom_cnn.keras")
save_history([cnn_history], "custom_cnn")


## 7. Train MobileNetV2 (frozen features, then fine-tune)

In [ ]:
from src.models.mobilenet_model import build_mobilenet_model, unfreeze_mobilenet_layers
mobile_train = make_dataset(train_df, "mobilenet", training=True)
mobile_val = make_dataset(val_df, "mobilenet")
CONTINUE_FROM_EXISTING = True
existing_path = Config.MODEL_PATH_MOBILENET_V2 if Config.MODEL_PATH_MOBILENET_V2.exists() else (REPO_DIR / "artifacts" / "checkpoints" / "mobilenetv2_best.keras")
if CONTINUE_FROM_EXISTING and existing_path.exists():
    print(f"Loading existing trained MobileNetV2 weights from {existing_path} to continue training...")
    mobilenet = tf.keras.models.load_model(str(existing_path))
    base = mobilenet.get_layer(Config.MOBILENET_BASE_MODEL_NAME)
    base.trainable = False
    for layer in base.layers:
        layer.trainable = False
    mobilenet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=Config.MOBILENET_INITIAL_LEARNING_RATE), loss="binary_crossentropy", metrics=["accuracy"])
else:
    mobilenet = build_mobilenet_model(input_shape=Config.INPUT_SHAPE)
phase1 = mobilenet.fit(mobile_train, validation_data=mobile_val, epochs=MOBILENET_WARMUP_EPOCHS, callbacks=get_callbacks("mobilenetv2_warmup"), class_weight=CLASS_WEIGHTS, verbose=1)
mobilenet = tf.keras.models.load_model(OUTPUT_DIR / "models" / "mobilenetv2_warmup_best.keras")
mobilenet = unfreeze_mobilenet_layers(mobilenet, num_layers_to_unfreeze=Config.MOBILENET_LAYERS_TO_UNFREEZE, fine_tune_lr=Config.MOBILENET_FINE_TUNE_LEARNING_RATE)
phase2 = mobilenet.fit(mobile_train, validation_data=mobile_val, epochs=MOBILENET_FINE_TUNE_EPOCHS, callbacks=get_callbacks("mobilenetv2_finetune"), class_weight=CLASS_WEIGHTS, verbose=1)
fine_tuned = tf.keras.models.load_model(OUTPUT_DIR / "models" / "mobilenetv2_finetune_best.keras")
warmup_model = tf.keras.models.load_model(OUTPUT_DIR / "models" / "mobilenetv2_warmup_best.keras")
warmup_val_loss = warmup_model.evaluate(mobile_val, verbose=0)[0]
finetune_val_loss = fine_tuned.evaluate(mobile_val, verbose=0)[0]
print(f"Best warmup validation loss: {warmup_val_loss:.4f}; fine-tuned: {finetune_val_loss:.4f}")
mobilenet = fine_tuned if finetune_val_loss <= warmup_val_loss else warmup_model
mobilenet.save(OUTPUT_DIR / "models" / "mobilenetv2_best.keras")
mobilenet.save(OUTPUT_DIR / "models" / "mobilenetv2.keras")
mobilenet.save(Config.MODEL_PATH_MOBILENET_V2)
save_history([phase1, phase2], "mobilenetv2")


## 8. Final validation and held-out test results

In [ ]:
all_results = {}
for model_key, model, normalization in (("custom_cnn", cnn, "standard"), ("mobilenetv2", mobilenet, "mobilenet")):
    validation_metrics = evaluate_split(model, val_df, normalization, "validation")
    test_metrics = evaluate_split(model, test_df, normalization, "test")
    all_results[model_key] = {"validation": validation_metrics, "test": test_metrics}
    (OUTPUT_DIR / "metrics" / f"{model_key}_results.json").write_text(json.dumps(all_results[model_key], indent=2), encoding="utf-8")
rows = [{"model": model_key, "split": split_name, **{k: metrics[k] for k in ("n", "accuracy", "precision", "recall", "f1_score", "roc_auc", "threshold")}} for model_key, split_results in all_results.items() for split_name, metrics in split_results.items()]
results_df = pd.DataFrame(rows)
display(results_df)
results_df.to_csv(OUTPUT_DIR / "metrics" / "validation_test_summary.csv", index=False)


## 9. Download trained models, plots, and result files

In [ ]:
import shutil
from google.colab import files
OUTPUT_ZIP = Path("/content/training_outputs.zip")
if OUTPUT_ZIP.exists(): OUTPUT_ZIP.unlink()
shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")), "zip", OUTPUT_DIR)
print(f"Created {OUTPUT_ZIP} ({OUTPUT_ZIP.stat().st_size / (1024**2):.1f} MiB)")
files.download(str(OUTPUT_ZIP))
